In [1]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
from transformers import AutoModelForCTC, Wav2Vec2Processor
import json
from tqdm import tqdm
import os
from textgrid import TextGrid
import editdistance
from VAD_chunk import *
from metrics import *

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import unicodedata

REF_MAPPING = {
    # Consonants
    "Z": "ʒ",
    "A":"a",
    "S": "ʃ",
    "R": "ʁ",
    "r": "ʁ",
    "N": "ŋ",
    "J": "ɲ",
    "H": "ɥ",
    "g": "ɡ",
    "Z=": "ʒ",

    # Vowels
    "E": "ɛ",
    "O": "ɔ",
    "2": "ø",
    "9": "œ",
    "@": "ə",

    # Nasals (SAMPA)
    "a~": "ɑ̃",
    "o~": "ɔ̃",
    "e~": "ɛ̃",
    "9~": "ɛ̃",
    "E": "ɛ",
    "m=": "m",
    "n=": "n",
    "9~": "ɛ̃",
}
NASAL_CANONICAL = {

    "ã": "ɑ̃",
    "ẽ": "ɛ̃",
    "ĩ": "ɛ̃",
    "ỹ": "ɛ̃",
    "œ̃": "ɛ̃",
    "ə̃": "ɛ̃",
}
HYP_PROJECTION = {

    # Multilingual vowel variants
    "ɪ": "i",
    "ʊ": "u",
    "ɨ": "i",
    "ɜ": "ə",
    "ʌ": "ɔ",
    "ɒ": "ɔ",

    # Rhotic variants
    "ɣ": "ʁ",
    "ɹ": "ʁ",
    "ɾ": "ʁ",

    # Lateral variant
    "ʎ": "l",

    # Foreign consonants
    "β": "b",
    "θ": "t",
    "c": "k",
    "ɑ": "a",
    "mʲ":"m",
    "ɟ": "ɲ",
}


In [3]:
def get_reference_alignments(textgrid_path,t=""):

    tg = TextGrid()
    tg.read(textgrid_path)

    ref_alignments = []

    # assuming tier name is "phones"
    tier = tg.getFirst(t)
    for interval in tier.intervals:

        phoneme = interval.mark.strip()

        if phoneme == "" or phoneme in ["sil", "sp", "spn"]:
            continue


        ref_alignments.append({
            "phoneme": phoneme,
            "start": interval.minTime,
            "end": interval.maxTime 
        })
    return ref_alignments
import Levenshtein

def align_sequences(ref, hyp):
    alignment = []
    ops = Levenshtein.editops(ref, hyp)

    ref_idx = hyp_idx = 0
    op_idx = 0

    while ref_idx < len(ref) or hyp_idx < len(hyp):

        if op_idx < len(ops):
            op_type, src_pos, dest_pos = ops[op_idx]

            if op_type == "delete" and src_pos == ref_idx:
                alignment.append((ref_idx, None))
                ref_idx += 1
                op_idx += 1
                continue

            elif op_type == "insert" and dest_pos == hyp_idx:
                alignment.append((None, hyp_idx))
                hyp_idx += 1
                op_idx += 1
                continue

            elif op_type == "replace" and \
                 src_pos == ref_idx and \
                 dest_pos == hyp_idx:
                alignment.append((ref_idx, hyp_idx))
                ref_idx += 1
                hyp_idx += 1
                op_idx += 1
                continue

        # equal case
        if ref_idx < len(ref) and hyp_idx < len(hyp):
            alignment.append((ref_idx, hyp_idx))
            ref_idx += 1
            hyp_idx += 1
        elif ref_idx < len(ref):
            alignment.append((ref_idx, None))
            ref_idx += 1
        elif hyp_idx < len(hyp):
            alignment.append((None, hyp_idx))
            hyp_idx += 1

    return alignment

In [4]:
VOWELS = set("aeiouyɛøœɔɑɨɪʊʌɒɜəɛ")

import unicodedata
import re
foreign_phonemes = {
    'β','θ','ɹ','ɾ','ɣ','ʌ','ʊ','ɪ','ɨ','ɨ̃','ɜ','ɒ','õ','ũ'
}
def normalize_phoneme_typaloc(ph):
    if ph is None:
        return None
    # Unicode normalization
    ph = unicodedata.normalize("NFC", ph)
    # enlever contenu [[...]]
    ph = re.sub(r"\[\[.*?\]\]", "", ph)

    # enlever crochets restants mal formés
    ph = re.sub(r"\[\[|\]\]", "", ph)

    # enlever contenu entre parenthèses
    ph = re.sub(r"\(.*?\)", "", ph)

    # enlever NONCORR
    ph = re.sub(r"\bNONCORR\b", "", ph)

    # nettoyer espaces
    ph = re.sub(r"\s+", " ", ph).strip()
    
    
    # enlever espaces multiples
    ph = re.sub(r"\s+", " ", ph).strip()
    ph =re.sub(r"\n.*", "", ph, flags=re.DOTALL)
    ph = ph.replace("yu","uy")
    ph = ph.replace("nn+yy","nn")
    ph = ph.replace("ei\t\t","ei")
    if "NB sur tDeb" in ph:
        ph="ei"
    ph = ph.replace("#a","a")
    ph = ph.replace("kk+","k")
    ph = re.sub(r"\*.*?\*", "", ph)
    ph = re.sub(r"\[\s*pause\s*\]", "", ph)
    ph = re.sub(r"\b\w*pause\w*\b", "", ph)
    
    if ph in ["_", "sil", "spn", "%", "?", "??","0","#", "=","euh","#erreur#"]:
        return None

    # Remove standalone combining marks
    if ph and all(unicodedata.combining(c) for c in ph):
        return None
    # --- Reference mapping ---
    if ph in asr_to_ipa.keys():
        ph = asr_to_ipa[ph]
    ph = ph.replace("dʒ","ʒ")
    return ph
def normalize_phoneme(ph, is_hyp=False):

    if ph is None:
        return None

    # 🔥 Strip first
    ph = ph.replace(" ", "")
    ph = re.sub(r"@t", "ə", ph)
    ph = re.sub(r"\?", "", ph)
    ph = re.sub(r"\@", "ə", ph)
    ph = ph.replace("<p:>", "")
    # Unicode normalization
    ph = unicodedata.normalize("NFC", ph)

    # Remove silence / junk
    if ph in {"_"," ", "sil", "spn", "%", "?", "0", "=", "fe~", "Ra~", "sjo~", "~e"}:
        return None

    # Remove standalone combining marks
    if ph and all(unicodedata.combining(c) for c in ph):
        return None

    # Reference mapping
    if not is_hyp and ph in REF_MAPPING:
        ph = REF_MAPPING[ph]

    # Hyp projection
    if is_hyp and ph in HYP_PROJECTION:
        ph = HYP_PROJECTION[ph]

    # Collapse multiple nasal marks
    ph = re.sub(r"\u0303+", "\u0303", ph)

    # Prevent nasalized consonants
    if len(ph) > 1:
        base = ph[0]
        if base not in VOWELS:
            ph = base

    # Canonical nasal mapping
    if ph in NASAL_CANONICAL:
        ph = NASAL_CANONICAL[ph]

    # Remove suprasegmentals
    ph = ph.replace("ː", "")

    # 🔥 Remove foreign AFTER projection
    if ph in foreign_phonemes:
        return None

    if ph == "":
        return None

    return ph


def clean_alignment_dict(alignment_list,flag="",is_hyp=False):
    """
    Normalize phonemes and remove empty or deleted ones.
    Keeps timestamps aligned.
    """
    
    cleaned = []
    
    for item in alignment_list:
        phoneme = item["phoneme"]

        if flag=="typaloc":
            phoneme_norm = normalize_phoneme_typaloc(phoneme)
        else:
            phoneme_norm = normalize_phoneme(phoneme, is_hyp=is_hyp)
        
        # Remove empty phonemes after normalization
        if phoneme_norm == "" or phoneme_norm is None:
            continue
        
        cleaned.append({
            "phoneme": phoneme_norm,
            "start": item["start"],
            "end": item["end"]
        })
    
    return cleaned


In [5]:
def match_alignments_lev(ref_alignments, hyp_alignments, ref_seq, hyp_seq,
                         ):

    alignment = align_sequences(ref_seq, hyp_seq)

    start_errors = []
    end_errors = []
    duration_errors = []
    mid_errors = []
    matched_pairs = []

    for ref_idx, hyp_idx in alignment:

        if ref_idx is None or hyp_idx is None:
            continue

        if ref_seq[ref_idx] != hyp_seq[hyp_idx]:
            continue

        r_start = ref_alignments[ref_idx]["start"]
        r_end   = ref_alignments[ref_idx]["end"]

        h_start = hyp_alignments[hyp_idx]["start"]
        h_end   = hyp_alignments[hyp_idx]["end"]

        mid_ref  = (r_start + r_end) / 2
        mid_pred = (h_start + h_end) / 2

        mid_error = abs(mid_ref - mid_pred)

        

        start_errors.append(abs(r_start - h_start))
        end_errors.append(abs(r_end - h_end))
        duration_errors.append(
            abs((r_end - r_start) - (h_end - h_start))
        )
        mid_errors.append(mid_error)
        matched_pairs.append((ref_idx, hyp_idx))

    return start_errors, end_errors, duration_errors, mid_errors, matched_pairs
import librosa

def get_phoneme_alignments(model, processor, audio_path):
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    wav = torch.from_numpy(audio)
    chunks = vad_chunk_with_timestamps(wav)
    device = next(model.parameters()).device
    blank_id = model.config.pad_token_id

    all_alignments = []
    full_phoneme_parts = []

    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_tensor = wav[start_sample:end_sample]

        inputs = processor(
            chunk_tensor.numpy(),
            sampling_rate=16000,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

        predicted_ids = torch.argmax(logits, dim=-1)[0]
        
        decoded = processor.batch_decode(predicted_ids.unsqueeze(0))[0]
        full_phoneme_parts.append(decoded.strip())

        num_frames     = logits.shape[1]
        chunk_duration = (end_sample - start_sample) / 16000
        frame_duration = chunk_duration / num_frames

        # ✅ Reset per chunk
        prev_id = None
        current_alignment = None

        for frame_idx, token_id in enumerate(predicted_ids.tolist()):
    
            if token_id == blank_id:
                prev_id = token_id
                continue
            
            phoneme = processor.decode([token_id])
            
            # Skip empty / space / word separator
            if phoneme in ["", " ", "|"]:
                prev_id = token_id
                continue
            
            # If this is a combining mark → attach to previous phoneme
            if unicodedata.combining(phoneme):
                if current_alignment is not None:
                    current_alignment["phoneme"] += phoneme
                prev_id = token_id
                continue
            
            if token_id != prev_id:
                if current_alignment is not None:
                    current_alignment["end"] = (
                        start_sample / 16000 + frame_idx * frame_duration
                    )
            
                start_time = start_sample / 16000 + frame_idx * frame_duration
                current_alignment = {
                    "phoneme": phoneme,
                    "start": start_time,
                    "end": None
                }
                all_alignments.append(current_alignment)
            
            prev_id = token_id


        # close last phoneme of this chunk using frame-based end
        if current_alignment is not None and current_alignment["end"] is None:
            current_alignment["end"] = (
                start_sample / 16000 + num_frames * frame_duration
            )
    full_phoneme_string = " ".join(full_phoneme_parts)
    return full_phoneme_string.strip(), all_alignments


In [6]:
import math, unicodedata, torch, torchaudio, numpy as np, soundfile as sf, librosa

WAVLM_STRIDE_SAMPLES = 320
WAVLM_STRIDE_S       = 0.02
MIN_SAMPLES          = 3200   # pad tiny chunks so the conv stack doesn't choke

def get_phoneme_alignments_w2v_ctcfa(model, processor, audio_path):
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    wav = torch.from_numpy(audio).float()

    chunks   = vad_chunk_with_timestamps(audio_path,max_chunk_duration=30)
    device   = next(model.parameters()).device
    blank_id = model.config.pad_token_id
    unk_id   = processor.tokenizer.unk_token_id

    all_alignments, full_phoneme_parts = [], []

    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_tensor = wav[start_sample:end_sample]
        real_samples = chunk_tensor.shape[-1]
        if real_samples == 0:
            continue
        if real_samples < MIN_SAMPLES:                       # pad short chunks
            chunk_tensor = torch.nn.functional.pad(chunk_tensor, (0, MIN_SAMPLES - real_samples))

        inputs = processor(chunk_tensor.numpy(), sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits                  # (1, T, V)

        # keep only frames belonging to the REAL (unpadded) audio
        valid_frames = max(1, min(math.ceil(real_samples / WAVLM_STRIDE_SAMPLES), logits.shape[1]))
        logits = logits[:, :valid_frames, :]

        predicted_ids = torch.argmax(logits, dim=-1)[0]

        # greedy collapse -> FA target + display string
        collapsed, prev = [], None
        for t in predicted_ids.tolist():
            if t != prev and t != blank_id:
                collapsed.append(t)
            prev = t
        full_phoneme_parts.append(" ".join(processor.tokenizer.convert_ids_to_tokens(collapsed)).strip())
        if not collapsed:
            continue

        # CTC forced alignment of the predicted sequence
        log_probs      = torch.log_softmax(logits.float(), dim=-1).cpu().contiguous()
        targets        = torch.tensor([collapsed], dtype=torch.int32)
        input_lengths  = torch.tensor([log_probs.shape[1]], dtype=torch.int32)
        target_lengths = torch.tensor([targets.shape[1]], dtype=torch.int32)
        try:
            aligned, _ = torchaudio.functional.forced_align(
                log_probs, targets, input_lengths, target_lengths, blank=blank_id)
        except Exception as e:
            print(f"  [forced_align skipped] {chunk['start']:.2f}-{chunk['end']:.2f}: {e}")
            continue
        aligned = aligned[0].tolist()

        # runs with start AND end frame (silence between phonemes stays unassigned)
        spans, run_id, run_start = [], None, 0
        for fi, tid in enumerate(aligned):
            if tid != run_id:
                if run_id is not None and run_id != blank_id:
                    spans.append((run_id, run_start, fi))
                run_id, run_start = tid, fi
        if run_id is not None and run_id != blank_id:
            spans.append((run_id, run_start, len(aligned)))

        off = start_sample / 16000
        chunk_aligns = []
        for token_id, sframe, eframe in spans:
            phoneme = processor.decode([token_id])
            if token_id == unk_id or phoneme in ["", " ", "[", "|"]:
                continue
            if len(phoneme) == 1 and unicodedata.combining(phoneme):
                if chunk_aligns:
                    chunk_aligns[-1]["phoneme"] += phoneme
                    chunk_aligns[-1]["end"] = off + eframe * WAVLM_STRIDE_S
                continue
            chunk_aligns.append({"phoneme": phoneme,
                                 "start": off + sframe * WAVLM_STRIDE_S,
                                 "end":   off + eframe * WAVLM_STRIDE_S})
        all_alignments.extend(chunk_aligns)

    return " ".join(full_phoneme_parts).strip(), all_alignments


In [17]:
import torchaudio  # new dependency: torchaudio >= 2.1 (for forced_align)

def get_phoneme_alignments_wavlm_ctcfa(model, feature_extractor, tokenizer, audio_path):
    """
    CTC forced-alignment variant (row 10).
    Same as the naive function except phoneme START frames come from
    torchaudio.functional.forced_align over the full CTC posteriors,
    not from the argmax best-path. Returned phoneme string == naive's,
    so PER is identical and boundary placement is the only variable.
    """
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    wav = torch.from_numpy(audio)

    chunks   = vad_chunk_with_timestamps(audio_path,max_chunk_duration=30)
    device   = next(model.parameters()).device
    blank_id = model.config.pad_token_id

    all_alignments     = []
    full_phoneme_parts = []

    for chunk in chunks:
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_tensor = wav[start_sample:end_sample]

        inputs = feature_extractor(
            chunk_tensor.numpy(), sampling_rate=16000, return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits                 # (1, T, V)

        predicted_ids = torch.argmax(logits, dim=-1)[0]      # greedy best path

        tokens = tokenizer.convert_ids_to_tokens(predicted_ids.tolist())
        probs = torch.softmax(logits[0], dim=-1)

    
        
        # --- unchanged: decoded string keeps PER identical to the naive system ---
        decoded = tokenizer.batch_decode(predicted_ids.unsqueeze(0))[0]

        full_phoneme_parts.append(decoded.strip())

        num_frames     = logits.shape[1]
        chunk_duration = (end_sample - start_sample) / 16000
        frame_duration = chunk_duration / num_frames

        # ---- CTC target = greedy-collapsed label sequence (repeats collapsed, blanks dropped) ----
        # identical to what the naive decoder emits, so both systems align the SAME phonemes
        targets_list, prev = [], None
        for t in predicted_ids.tolist():
            if t != prev and t != blank_id:
                targets_list.append(t)
            prev = t
        if not targets_list:
            continue                                        # silence / blank-only chunk

        # ---- forced-align that sequence against the full posteriors ----
        log_probs      = torch.log_softmax(logits.float(), dim=-1).cpu().contiguous()
        targets        = torch.tensor([targets_list], dtype=torch.int32)
        input_lengths  = torch.tensor([log_probs.shape[1]], dtype=torch.int32)
        target_lengths = torch.tensor([targets.shape[1]], dtype=torch.int32)
        try:
            aligned, _ = torchaudio.functional.forced_align(
                log_probs, targets, input_lengths, target_lengths, blank=blank_id
            )
        except Exception as e:                              # e.g. T too short for the token count
            print(f"[forced_align skipped] {audio_path} "
                  f"{chunk['start']:.2f}-{chunk['end']:.2f}: {e}")
            continue
        aligned = aligned[0].tolist()                       # token id per frame (blanks = blank_id)

        # ---- merge frames into one span per token instance; keep its onset frame ----
        spans, run_id, run_start = [], None, 0
        for f, tid in enumerate(aligned):
            if tid != run_id:
                if run_id is not None and run_id != blank_id:
                    spans.append((run_id, run_start))
                run_id, run_start = tid, f
        if run_id is not None and run_id != blank_id:
            spans.append((run_id, run_start))

        # ---- spans -> alignments (same skip / combining-mark rules as the naive version) ----
        chunk_aligns = []
        for token_id, sframe in spans:
            phoneme = tokenizer.decode([token_id])
            if token_id == tokenizer.unk_token_id or phoneme in ["", " ", "[","|"]:
                continue
            if len(phoneme) == 1 and unicodedata.combining(phoneme):
                if chunk_aligns:
                    chunk_aligns[-1]["phoneme"] += phoneme
                continue
            start_time = start_sample / 16000 + sframe * frame_duration
            chunk_aligns.append({"phoneme": phoneme, "start": start_time, "end": None})
        # contiguous ends (end_i = start_{i+1}; last = chunk end) — same convention as naive
        for i in range(len(chunk_aligns) - 1):
            chunk_aligns[i]["end"] = chunk_aligns[i + 1]["start"]
        if chunk_aligns:
            chunk_aligns[-1]["end"] = start_sample / 16000 + num_frames * frame_duration

        all_alignments.extend(chunk_aligns)

    full_phoneme_string = " ".join(full_phoneme_parts)
    return full_phoneme_string.strip(), all_alignments

In [32]:
import math
import torch
import numpy as np
import soundfile as sf
import librosa
import unicodedata
import torchaudio  # >= 2.1 for forced_align

# Whisper encoder: 10 ms mel hop * 2 (conv stride-2) = 20 ms = 320 samples @16 kHz
WHISPER_FRAME_STRIDE_S       = 0.02
WHISPER_FRAME_STRIDE_SAMPLES = 320

def get_phoneme_alignments_whisper_ctcfa(model, feature_extractor, tokenizer,
                                          audio_path, reference=None):
    """
    CTC forced-alignment for a Whisper-encoder + CTC-head phoneme model.
    Now also prints, per chunk, the clean predicted phoneme string. The reference
    (full utterance) is printed once, since VAD chunks have no per-chunk reference.
    """
    audio, sr = sf.read(audio_path)
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sr != 16000:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=16000)
    wav = torch.from_numpy(audio).float()

    chunks = vad_chunk_with_timestamps(audio_path)
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    blank_id = getattr(model.config, "pad_token_id", None)
    if blank_id is None:
        blank_id = tokenizer.pad_token_id

    if reference is not None:
        print(f"REF (utterance): {reference}")

    all_alignments, full_phoneme_parts = [], []

    for ci, chunk in enumerate(chunks):
        start_sample = int(chunk["start"] * 16000)
        end_sample   = int(chunk["end"]   * 16000)
        chunk_tensor = wav[start_sample:end_sample]
        if chunk_tensor.numel() == 0:
            continue

        inputs = feature_extractor(
            chunk_tensor.numpy(), sampling_rate=16000, return_tensors="pt",
        )
        input_features = inputs.input_features.to(device).to(dtype)

        with torch.no_grad():
            logits = model(input_features=input_features).logits     # (1, ~1500, V)

        valid_frames = max(1, math.ceil(chunk_tensor.shape[0] / WHISPER_FRAME_STRIDE_SAMPLES))
        valid_frames = min(valid_frames, logits.shape[1])
        logits = logits[:, :valid_frames, :]

        predicted_ids = torch.argmax(logits, dim=-1)[0]

        # ---- clean CTC collapse for display (drop repeats AND blanks) ----
        collapsed, prev = [], None
        for t in predicted_ids.tolist():
            if t != prev and t != blank_id:
                collapsed.append(t)
            prev = t
        pred_str = " ".join(tokenizer.convert_ids_to_tokens(collapsed))

        full_phoneme_parts.append(pred_str.strip())

        num_frames     = logits.shape[1]
        frame_duration = WHISPER_FRAME_STRIDE_S

        # ---- CTC target for forced-align (same collapse) ----
        targets_list = collapsed                         # reuse the collapse above
        if not targets_list:
            continue

        log_probs      = torch.log_softmax(logits.float(), dim=-1).cpu().contiguous()
        targets        = torch.tensor([targets_list], dtype=torch.int32)
        input_lengths  = torch.tensor([log_probs.shape[1]], dtype=torch.int32)
        target_lengths = torch.tensor([targets.shape[1]], dtype=torch.int32)
        try:
            aligned, _ = torchaudio.functional.forced_align(
                log_probs, targets, input_lengths, target_lengths, blank=blank_id
            )
        except Exception as e:
            print(f"  [forced_align skipped] chunk {ci} "
                  f"{chunk['start']:.2f}-{chunk['end']:.2f}: {e}")
            continue
        aligned = aligned[0].tolist()

        spans, run_id, run_start = [], None, 0
        for f, tid in enumerate(aligned):
            if tid != run_id:
                if run_id is not None and run_id != blank_id:
                    spans.append((run_id, run_start))
                run_id, run_start = tid, f
        if run_id is not None and run_id != blank_id:
            spans.append((run_id, run_start))

        chunk_aligns = []
        for token_id, sframe in spans:
            phoneme = tokenizer.decode([token_id])
            if token_id == tokenizer.unk_token_id or phoneme in ["", " ", "[", "|"]:
                continue
            if len(phoneme) == 1 and unicodedata.combining(phoneme):
                if chunk_aligns:
                    chunk_aligns[-1]["phoneme"] += phoneme
                continue
            start_time = start_sample / 16000 + sframe * frame_duration
            chunk_aligns.append({"phoneme": phoneme, "start": start_time, "end": None})
        for i in range(len(chunk_aligns) - 1):
            chunk_aligns[i]["end"] = chunk_aligns[i + 1]["start"]
        if chunk_aligns:
            chunk_aligns[-1]["end"] = start_sample / 16000 + num_frames * frame_duration

        all_alignments.extend(chunk_aligns)

    full_phoneme_string = " ".join(full_phoneme_parts).strip()
    return full_phoneme_string, all_alignments

In [7]:
"""
WhisperX-style VAD chunking for the Whisper-encoder + CTC phoneme model.

Built on your approach: uses whisperx.vads.pyannote.load_vad_model, reads the
RAW per-frame VAD scores, and binarizes them with WhisperX's onset/offset
thresholds. The goal is unchanged from the original vad_chunk_with_timestamps:
return a list of {"start", "end"} (seconds, original time) chunks, each <= 30 s,
ready for the inference loop.

Two stages:
  1. VAD + binarize  -> WhisperX raw scores -> speech segments (hysteresis: go
                        active above `onset`, inactive below `offset`).
  2. cut & merge     -> pack segments into <= chunk_size windows, KEEPING internal
                        pauses inside a window. A single segment longer than
                        chunk_size is split at its QUIETEST frame (real WhisperX
                        behaviour, possible here because we kept the raw scores),
                        so nothing ever hits Whisper's 30 s truncation.

Only load_whisperx_vad() touches whisperx, so the rest is importable/testable
offline without it.
"""

import numpy as np
import torch
from whisperx.vads.pyannote import load_vad_model
# WhisperX defaults
ONSET = 0.5
OFFSET = 0.363


def load_whisperx_vad(wav):
    vad_pipeline = load_vad_model(
    device="cuda",
    token=os.environ["HF_TOKEN"])
    vad_scores = vad_pipeline(wav)
    scores = vad_scores.data[:, 0]
    frames = vad_scores.sliding_window
    times = [frames[i].middle for i in range(len(scores))]
    return scores, times

def _binarize(scores, times, onset=ONSET, offset=OFFSET):
    """Hysteresis binarization -> list of (start_s, end_s) speech segments."""
    segments = []
    is_active = scores[0] > onset
    start = times[0] if is_active else None
    for t, sc in zip(times[1:], scores[1:]):
        if is_active:
            if sc < offset:
                segments.append((start, t))
                is_active = False
        else:
            if sc > onset:
                start = t
                is_active = True
    if is_active:
        segments.append((start, times[-1]))
    return segments


def _split_long_by_score(seg_start, seg_end, times, scores, chunk_size):
    """Split a > chunk_size segment at the quietest frame in each window.

    Mirrors WhisperX: when a segment exceeds max_duration, cut at the lowest
    detection score in the second half rather than at a hard time boundary, so
    the cut lands on minimally-active speech.
    """
    pieces, cur = [], seg_start
    while seg_end - cur > chunk_size:
        lo, hi = cur + chunk_size * 0.5, cur + chunk_size
        idx = np.where((times >= lo) & (times <= hi))[0]
        cut = times[idx[np.argmin(scores[idx])]] if len(idx) else cur + chunk_size
        pieces.append((cur, cut))
        cur = cut
    pieces.append((cur, seg_end))
    return pieces


def merge_chunks(segments, times, scores, chunk_size=20.0):
    """WhisperX cut & merge -> list of {"start","end"} chunks, each <= chunk_size."""
    times = np.asarray(times, dtype=float)
    scores = np.asarray(scores, dtype=float)
    split = []
    for s, e in segments:
        if e - s > chunk_size:
            split.extend(_split_long_by_score(s, e, times, scores, chunk_size))
        else:
            split.append((s, e))

    if not split:
        return []

    merged = []
    curr_start, curr_end = split[0]
    for s, e in split[1:]:
        if e - curr_start > chunk_size and curr_end - curr_start > 0:
            merged.append({"start": curr_start, "end": curr_end})
            curr_start = s
        curr_end = e
    merged.append({"start": curr_start, "end": curr_end})
    return merged


def vad_chunk_with_timestamps(
    wav,
    sampling_rate=16000,
    max_chunk_duration=8.0,
    onset=ONSET,
    offset=OFFSET,
):
    """Drop-in replacement. Same return type as the old rVAD version.

    wav               : torch.Tensor (1D, 16 kHz)  -- or a file path
    vad_model         : object from load_whisperx_vad() (load it once, reuse it)
    max_chunk_duration: keep <= 30.0 for Whisper's hard cap
    """
    scores, times = load_whisperx_vad(wav)
    segments = _binarize(scores, times, onset, offset)
    return merge_chunks(segments, times, scores, chunk_size=max_chunk_duration)

/home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


In [24]:
MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wav2vec2-french-phonemizer"
#MODEL_ID = "/vol/experiments3/imbenamor/TAPAS-FRAIS/models/wavlm_finetuned_big/checkpoint-11000"

model = AutoModelForCTC.from_pretrained(MODEL_ID)
processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)

device = "cpu"
model = model.to(device)
model.eval()

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2GroupNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (activation): GELUActivation()
          (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
        )
        (1-4): 4 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2NoLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projection): Linear(in_features=512, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder)

In [34]:
import torch, librosa
from transformers import WavLMForCTC, Wav2Vec2FeatureExtractor, Wav2Vec2PhonemeCTCTokenizer

#CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme/checkpoint-268600"
CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large/checkpoint-253984"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/src/finetuning/wavlm-fr-phoneme-large"          # where vocab.json / tokenizer were saved

device = "cuda"
model = WavLMForCTC.from_pretrained(CKPT).to(device).eval()
feat  = Wav2Vec2FeatureExtractor.from_pretrained(CKPT)   # also present in CKPT
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"
model = model.to(device)
model.eval()

WavLMForCTC(
  (wavlm): WavLMModel(
    (feature_extractor): WavLMFeatureEncoder(
      (conv_layers): ModuleList(
        (0): WavLMLayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x WavLMLayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): WavLMFeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
 

In [35]:
#whisper
from transformers import WhisperFeatureExtractor, Wav2Vec2PhonemeCTCTokenizer
#whisper model
from transformers import (
    WhisperConfig,
    WhisperFeatureExtractor,
    WhisperModel,
    WhisperPreTrainedModel,
    Wav2Vec2PhonemeCTCTokenizer,
    TrainingArguments,
    Trainer,
)
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers.models.whisper.modeling_whisper import WhisperEncoder
from transformers.modeling_outputs import CausalLMOutput
class WhisperEncoderForCTC(WhisperPreTrainedModel):
    """Whisper encoder with a fresh linear CTC head; decoder is not instantiated."""
    config_class = WhisperConfig
    main_input_name = "input_features"
    def __init__(self, config):
        super().__init__(config)
        self.encoder = WhisperEncoder(config)
        self.dropout = nn.Dropout(getattr(config, "final_dropout", 0.0))
        # config.vocab_size is overridden at load time to match the phoneme vocab.
        self.lm_head = nn.Linear(config.d_model, config.vocab_size)
        self.post_init()
    def freeze_conv_frontend(self):
        for p in self.encoder.conv1.parameters():
            p.requires_grad = False
        for p in self.encoder.conv2.parameters():
            p.requires_grad = False
    def forward(self, input_features=None, labels=None, attention_mask=None, **kwargs):
        encoder_out = self.encoder(input_features).last_hidden_state  # (B, 1500, d_model)
        logits = self.lm_head(self.dropout(encoder_out))               # (B, 1500, V)
        loss = None
        if labels is not None:
            # CTC expects (T, B, V) log-probs in float32.
            log_probs = F.log_softmax(logits, dim=-1, dtype=torch.float32).transpose(0, 1)
            input_lengths = torch.full(
                (logits.shape[0],), logits.shape[1],
                dtype=torch.long, device=logits.device,
            )
            labels_mask = labels >= 0
            target_lengths = labels_mask.sum(-1)
            flat_targets = labels.masked_select(labels_mask)
            # cuDNN CTC has stricter shape constraints; safer to use the native impl.
            with torch.backends.cudnn.flags(enabled=False):
                loss = F.ctc_loss(
                    log_probs, flat_targets, input_lengths, target_lengths,
                    blank=self.config.pad_token_id,
                    reduction="mean", zero_infinity=True,
                )
        return CausalLMOutput(loss=loss, logits=logits)
CKPT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme/checkpoint-31327"
ROOT = "/vol/experiments3/imbenamor/TAPAS-FRAIS/whisper-fr-phoneme"
device = "cuda"

model = WhisperEncoderForCTC.from_pretrained(CKPT).to(device).eval()
feat  = WhisperFeatureExtractor.from_pretrained(CKPT)
tok   = Wav2Vec2PhonemeCTCTokenizer.from_pretrained(ROOT)
tok.unk_token = "[UNK]"
tok.pad_token = "[PAD]"

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'WhisperTokenizer'. 
The class this function is called from is 'Wav2Vec2PhonemeCTCTokenizer'.


In [9]:
def extract_phoneme_sequence(alignment_list):
    return [item["phoneme"] for item in alignment_list]
import soundfile as sf

import soundfile as sf

def get_ref_intervals(clean_ref, audio_path, threshold=0.5):
    audio, sr = sf.read(audio_path)
    audio_duration = len(audio) / sr
    ref_last = clean_ref[-1]["end"]
    ref_offset = clean_ref[0]["start"]

    if ref_last > audio_duration + threshold:
        print(f"  Session offset detected: ref_last={ref_last:.2f}s, "
              f"audio={audio_duration:.2f}s, offset={ref_offset:.3f}s")
        return [{
            "phoneme": item["phoneme"],
            "start": item["start"] - ref_offset,
            "end": item["end"] - ref_offset
        } for item in clean_ref]
    else:
        return list(clean_ref)  # ← this was missing

def get_hyp_intervals(clean_hyp):
    """
    Hyp timestamps are always audio-relative (CTC frame * stride).
    No correction needed.
    """
    return list(clean_hyp)

In [1]:
from jiwer import process_words
from collections import defaultdict
from pathlib import Path

audio_dir = "/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Data_Mons/Description"
textgrid_dir = Path("/vol/corpora/TAPAS_FRAIS/Data_Partagees_Mons/Mons TextGrid verifie + 50 ans" )
ref_files = textgrid_dir.glob("*") 

ref_dict = {f.stem:f for f in ref_files}


In [28]:
import pickle
alignment_store = {}

for f in ref_dict.keys():
    audio_path = os.path.join(audio_dir, f + ".wav")
    textgrid_path = ref_dict[f]
    
    pred_phonemes, pred_alignments = get_phoneme_alignments_wavlm_ctcfa(model, feat,tok, audio_path)
    ref_alignments = get_reference_alignments(textgrid_path, t="MAU")
    
    clean_hyp = clean_alignment_dict(pred_alignments, is_hyp=True)
    clean_ref = clean_alignment_dict(ref_alignments, is_hyp=False)
    
    # Fix session offset in ref if needed — no hyp information used
    ref_intervals = get_ref_intervals(clean_ref, audio_path, threshold=0.5)
    hyp_intervals = get_hyp_intervals(clean_hyp)

    alignment_store[f] = {
        "ref_intervals": ref_intervals,
        "hyp_intervals": hyp_intervals,
        "ref_seq": extract_phoneme_sequence(clean_ref),
        "hyp_seq": extract_phoneme_sequence(clean_hyp),
    }
    


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx/assets/pytorch_model.bin`
/tmp/ipykernel_2393135/715686889.py:69: UserWarning: torchaudio.functional._alignment.forced_align has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  aligned, _ = torchaudio.functional.forced_align(
Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.5. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint ../../../../../../home/imbenamor/miniconda3/envs/pyannote_env/lib/python3.10/site-packages/whisperx

In [29]:
with open("ctc_results/alignment_wavlm_mon.pkl", "wb") as f:
    pickle.dump(alignment_store, f)

In [30]:
from metrics_alignment import *

with open("ctc_results/alignment_wavlm_mon.pkl", "rb") as f:
    alignment_store = pickle.load(f)
compute_metrics(alignment_store, "ctc_results/metrics++_wavlm_mon.csv", per_phoneme_csv=None)
#metrics(alignment_store,"ctc_results/results_tapas.csv")


,group,style,K,AAS (ms),Median_start (ms),Mean_start (ms),Median_end (ms),Mean_end (ms),P90 (ms),%>50ms,...,Median_dur (ms),%dur>50ms,N_ref,N_hyp,PER (%),Deletions,Insertions,F1@0ms (%),F1@20ms (%),F1@50ms (%)
0,MO_H_ER48_2017_01_28_ModuleDescription_descrip...,ALL,230,95.890433,40.190944,58.754928,41.861413,133.025938,104.081655,40.652174,...,30.041551,34.347826,278,255,17.985612,25,2,0.00000,33.020638,67.542214
1,MO_F_ER19_2016_04_10_ModuleDescription_descrip...,ALL,192,86.765179,40.504524,47.424972,47.584010,126.105386,119.437651,43.229167,...,40.115128,44.791667,207,209,9.661836,3,5,0.00000,31.250000,69.230769
2,MO_H_SD36_2016_06_23_ModuleDescription_descrip...,ALL,356,106.046627,50.707927,66.346454,55.460962,145.746800,138.930594,53.932584,...,30.126915,35.955056,402,385,13.930348,27,10,0.00000,25.412961,61.499365
3,MO_H_SD11_2016_04_25_ModuleDescription_descrip...,ALL,537,83.853762,42.815186,49.832549,44.272127,117.874976,112.175854,42.737430,...,30.123355,37.616387,595,590,13.109244,25,20,0.00000,33.924051,70.210970
4,MO_H_AV18_2016_04_18_ModuleDescription_descrip...,ALL,186,85.369765,48.959044,52.893042,50.235112,117.846487,117.811519,49.193548,...,30.064846,34.408602,208,201,12.980769,12,5,0.00000,25.427873,61.613692
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65,MO_H_SD30_2016_05_21_ModuleDescription_descrip...,ALL,342,78.323079,33.092772,52.743037,36.791874,103.903121,89.324324,28.654971,...,27.855880,23.391813,398,395,20.351759,28,25,0.00000,34.552333,76.670870
66,MO_F_AV21_2016_04_18_ModuleDescription_descrip...,ALL,321,101.517202,42.468957,68.537146,47.733103,134.497259,121.282285,44.392523,...,30.032573,27.102804,365,374,20.000000,20,29,0.00000,28.416779,66.305819
67,MO_H_ER49_2017_01_28_ModuleDescription_descrip...,ALL,197,60.200939,34.439858,38.325667,37.517689,82.076210,84.416274,26.903553,...,30.035377,26.395939,229,248,27.074236,11,30,0.00000,33.542977,74.213836
68,STYLE_ALL,ALL,26280,81.914526,41.543921,53.633276,44.237126,110.195777,110.879779,41.849315,...,30.053354,32.404871,30592,29466,17.076360,2038,912,0.00333,30.630391,69.269706


In [31]:
import csv

def dict_to_csv(data_dict, output_path='phonemes.csv'):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['filename', 'predicted_phonemes'])
        for filename, info in data_dict.items():
            phonemes = ' '.join([interval['phoneme'] for interval in info['hyp_intervals']])
            writer.writerow([filename, phonemes])
with open("ctc_results/alignment_wavlm_mon.pkl", "rb") as f:
    alignment_store = pickle.load(f)
dict_to_csv(alignment_store, 'pred_wavlm_mon.csv')

In [10]:

"""corpus_per =  (total_S + total_D + total_I) / total_N
precision_20 = (
    total_TP_20 / (total_TP_20 + total_FP_20)
    if (total_TP_20 + total_FP_20) > 0 else 0
)

recall_20 = (
    total_TP_20 / (total_TP_20 + total_FN_20)
    if (total_TP_20 + total_FN_20) > 0 else 0
)

f1_20 = (
    2 * precision_20 * recall_20 / (precision_20 + recall_20)
    if (precision_20 + recall_20) > 0 else 0
)

print("Precision@50ms:", precision_20 * 100)
print("Recall@50ms:", recall_20 * 100)
print("F1@50ms:", f1_20 * 100)


if len(all_boundary_errors) > 0:
    #flat_errors = [e for sublist in all_boundary_errors for e in sublist]
    errors = np.array(all_boundary_errors)
    mp_errors = np.array(mid_BE)
    print("GLOBAL max error (sec):", np.max(errors))
    print("GLOBAL median error (sec):", np.median(errors))

    print("GLOBAL max miderror (sec):", np.max(mp_errors))
    print("GLOBAL median miderror (sec):", np.median(mp_errors))
    mean_boundary_error = np.mean(errors)
    median_boundary_error = np.median(errors)

    within_20ms = np.mean(errors <= 0.02) * 100
    within_50ms = np.mean(errors <= 0.05) * 100

    mean_boundary_mperror = np.mean(mp_errors)
    median_boundary_mperror = np.median(mp_errors)

    mp_within_20ms = np.mean(mp_errors <= 0.02) * 100
    mp_within_50ms = np.mean(mp_errors <= 0.05) * 100


final_results = pd.DataFrame([{
    "Precision@50ms (%)": precision_20 * 100,
    "Recall@50ms (%)": recall_20 * 100,
    "F1@50ms (%)": f1_20 * 100,
    "Corpus PER (%)": corpus_per * 100,
    "Mean_midpoint_error (ms)": mean_boundary_mperror * 1000,
    "Median_midpoint_error (ms)": median_boundary_mperror * 1000,
    "% within 20ms (midpoint)": mp_within_20ms,
    "% within 50ms (midpoint)": mp_within_50ms
}])

final_results"""

Precision@50ms: 73.34437645786146
Recall@50ms: 70.92050209205021
F1@50ms: 72.11207684509662
GLOBAL max error (sec): 6.073156565656561
GLOBAL median error (sec): 0.051717612809316194
GLOBAL max miderror (sec): 6.078459595959586
GLOBAL median miderror (sec): 0.050373244893791025


,Precision@50ms (%),Recall@50ms (%),F1@50ms (%),Corpus PER (%),Mean_midpoint_error (ms),Median_midpoint_error (ms),% within 20ms (midpoint),% within 50ms (midpoint)
0,73.344376,70.920502,72.112077,18.197568,290.470953,50.373245,22.245997,49.730618


In [11]:
"""final_results.to_csv("results_tapas.csv")

df_pred = pd.DataFrame(all_predicted_segments)
df_pred.to_csv("all_predicted_alignments_tapas.csv", index=False)
final_results"""

,Precision@50ms (%),Recall@50ms (%),F1@50ms (%),Corpus PER (%),Mean_midpoint_error (ms),Median_midpoint_error (ms),% within 20ms (midpoint),% within 50ms (midpoint)
0,73.344376,70.920502,72.112077,18.197568,290.470953,50.373245,22.245997,49.730618
